# AdaBoost (Adaptive Boosting)

**Boosting**: a diferencia del *averaging* (bagging, random forest), los modelos se entrenan de forma **secuencial** y cada uno aprende de los errores del anterior. AdaBoost aumenta el peso de las observaciones que el modelo previo predijo mal.

Aca lo usamos en regresion (`AdaBoostRegressor`) sobre `Hitters`, con **regresion lineal** como estimador base, para predecir el **log del salario**. Comparamos contra la regresion lineal sola.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import AdaBoostRegressor
from sklearn.metrics import mean_squared_error

## Datos

`Hitters` sin nulos, solo variables numericas. Target = `log(Salary)`.

In [ ]:
data_complete = pd.read_csv("../datasets/Hitters.csv").dropna()

data_columns=['AtBat','Hits','HmRun','Runs','RBI','Walks', 'Years', 'CAtBat', 'CHits', 'CHmRun', 'CRuns', 'CRBI', 'CWalks',
                'PutOuts', 'Assists', 'Errors', 'Salary']
data = data_complete.loc[:, data_columns]

X = data.drop("Salary", axis=1)
y = np.log(data.Salary)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=127)

In [ ]:
scaler = StandardScaler()
X_train_scl = scaler.fit_transform(X_train)
X_test_scl = scaler.transform(X_test)

## Referencia: regresion lineal sola

In [ ]:
base_regressor = LinearRegression()
fit_base = base_regressor.fit(X_train_scl, y_train)
predict_base = fit_base.predict(X_test_scl)
performance_base = mean_squared_error(y_test, predict_base)
print(performance_base)

## AdaBoost

`AdaBoostRegressor` entrena 200 regresiones lineales en secuencia (`learning_rate=0.8` pondera el aporte de cada una). Reportamos el MSE sobre test.

In [ ]:
boost_linreg = AdaBoostRegressor(estimator=base_regressor, n_estimators=200, learning_rate=0.8, loss='linear', random_state=127)

In [ ]:
boost_linreg.fit(X_train_scl, y_train)

In [ ]:
prediction = boost_linreg.predict(X_test_scl)
performance = mean_squared_error(y_test, prediction)
performance

MSE base ~0.414 vs AdaBoost ~0.439. Con un estimador base **lineal** (bajo sesgo pero poca varianza) el boosting no aporta: AdaBoost brilla combinando estimadores **debiles** (arboles poco profundos). Con regresion lineal como base, apenas empata (o empeora) al modelo simple.